# 6. Classification — Logistic Regression

**Scratch implementation**: Sigmoid activation + Binary Cross-Entropy loss + Gradient Descent  
**sklearn confirmation**: `LogisticRegression`

### Theory
The logistic regression model predicts the probability that the next candle is Up:

$$\hat{p} = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}$$

Loss (Binary Cross-Entropy):

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i) \right]$$

Gradient Descent weight update (derived from the chain rule):

$$w \leftarrow w - \alpha \cdot \frac{1}{n} X^T(\hat{p} - y)$$
$$b \leftarrow b - \alpha \cdot \frac{1}{n} \sum(\hat{p} - y)$$

## 1. Imports, Setup & Shared Utilities

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve)

BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..'))
TRAIN_FILE    = os.path.join(BASE_DIR, 'data', 'final_training_data_classification_train.csv')
TEST_FILE     = os.path.join(BASE_DIR, 'data', 'final_training_data_classification_test.csv')
FEATURES_FILE = os.path.join(BASE_DIR, 'selected_features_classification.json')
SCALER_FILE   = os.path.join(BASE_DIR, 'models', 'scaler_classification.pkl')
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
RESULTS_DIR   = os.path.join(BASE_DIR, 'results')
PLOTS_DIR     = os.path.join(BASE_DIR, 'plots')
RESULTS_CSV   = os.path.join(RESULTS_DIR, 'classification_results.csv')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

def compute_metrics(model_name, y_true, y_pred, y_prob=None):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    accuracy    = (tp + tn) / (tp + tn + fp + fn)
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1          = 2*precision*sensitivity/(precision+sensitivity) if (precision+sensitivity) > 0 else 0.0
    auc         = roc_auc_score(y_true, y_prob) if y_prob is not None else float('nan')
    print(f"\n{'='*55}\n  {model_name}\n{'='*55}")
    print(f"  Confusion Matrix : TN={tn}  FP={fp}  FN={fn}  TP={tp}")
    print(f"  Accuracy         : {accuracy:.4f}")
    print(f"  Precision        : {precision:.4f}")
    print(f"  Sensitivity      : {sensitivity:.4f}")
    print(f"  Specificity(TNR) : {specificity:.4f}")
    print(f"  F1-Score         : {f1:.4f}")
    print(f"  ROC-AUC          : {auc:.4f}")
    print(f"{'='*55}")
    return {'model': model_name, 'accuracy': round(accuracy,4), 'precision': round(precision,4),
            'sensitivity': round(sensitivity,4), 'specificity': round(specificity,4),
            'tnr': round(specificity,4), 'f1': round(f1,4),
            'roc_auc': round(auc,4) if not np.isnan(auc) else None,
            'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)}

def save_results(metrics_dict):
    row = pd.DataFrame([metrics_dict])
    if os.path.exists(RESULTS_CSV):
        existing = pd.read_csv(RESULTS_CSV)
        existing = existing[existing['model'] != metrics_dict['model']]
        row = pd.concat([existing, row], ignore_index=True)
    row.to_csv(RESULTS_CSV, index=False)
    print(f"Results saved → {RESULTS_CSV}")

def plot_confusion_matrix(y_true, y_pred, model_name, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Down (0)','Up (1)'], yticklabels=['Down (0)','Up (1)'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix — {model_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

print("Utilities ready.")

## 2. Load & Scale Data

In [ ]:
with open(FEATURES_FILE) as f:
    features = json.load(f)

with open(SCALER_FILE, 'rb') as f:
    scaler = pickle.load(f)

train_df = pd.read_csv(TRAIN_FILE)
test_df  = pd.read_csv(TEST_FILE)

# Features are already saved scaled in the CSVs from Phase 3,
# but we apply the scaler again to be explicit and consistent.
X_train = scaler.transform(train_df[features].values)
y_train = train_df['target_class'].values
X_test  = scaler.transform(test_df[features].values)
y_test  = test_df['target_class'].values

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"Features: {features}")

## 3. Scratch Implementation — Logistic Regression via Gradient Descent

In [ ]:
class LogisticRegressionScratch:
    """
    Binary Logistic Regression from scratch.
    Uses sigmoid activation, binary cross-entropy loss, and gradient descent.
    """
    def __init__(self, learning_rate=0.01, n_epochs=1000):
        self.lr       = learning_rate
        self.n_epochs = n_epochs
        self.w        = None
        self.b        = None
        self.loss_history = []

    @staticmethod
    def _sigmoid(z):
        # Clip to avoid overflow in exp
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for epoch in range(self.n_epochs):
            # Forward pass
            z    = X @ self.w + self.b
            p    = self._sigmoid(z)

            # Binary cross-entropy loss
            eps  = 1e-15   # numerical stability
            loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
            self.loss_history.append(loss)

            # Gradients (derived via chain rule)
            error   = p - y
            dw      = (X.T @ error) / n_samples
            db      = np.mean(error)

            # Update weights
            self.w -= self.lr * dw
            self.b -= self.lr * db

            if (epoch + 1) % 100 == 0:
                print(f"  Epoch {epoch+1:4d}/{self.n_epochs}  |  Loss: {loss:.6f}")

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


# Train
lr_scratch = LogisticRegressionScratch(learning_rate=0.1, n_epochs=500)
lr_scratch.fit(X_train, y_train)

In [ ]:
# Loss curve
plt.figure(figsize=(10, 4))
plt.plot(lr_scratch.loss_history, color='steelblue', linewidth=1.5)
plt.xlabel('Epoch'); plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Logistic Regression (Scratch) — Training Loss Curve', fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
loss_plot = os.path.join(PLOTS_DIR, 'lr_scratch_loss_curve.png')
plt.savefig(loss_plot, dpi=150, bbox_inches='tight')
plt.show()

# Evaluate
y_pred_scratch = lr_scratch.predict(X_test)
y_prob_scratch = lr_scratch.predict_proba(X_test)
metrics_scratch = compute_metrics('Logistic Regression (Scratch)', y_test, y_pred_scratch, y_prob_scratch)
plot_confusion_matrix(y_test, y_pred_scratch, 'Logistic Regression (Scratch)',
                      os.path.join(PLOTS_DIR, 'cm_lr_scratch.png'))

## 4. sklearn Confirmation

In [ ]:
lr_sk = LogisticRegression(max_iter=1000, random_state=42)
lr_sk.fit(X_train, y_train)

y_pred_sk = lr_sk.predict(X_test)
y_prob_sk = lr_sk.predict_proba(X_test)[:, 1]

metrics_sk = compute_metrics('Logistic Regression (sklearn)', y_test, y_pred_sk, y_prob_sk)
plot_confusion_matrix(y_test, y_pred_sk, 'Logistic Regression (sklearn)',
                      os.path.join(PLOTS_DIR, 'cm_lr_sklearn.png'))

print("\nScratch vs sklearn accuracy comparison:")
print(f"  Scratch : {metrics_scratch['accuracy']:.4f}")
print(f"  sklearn : {metrics_sk['accuracy']:.4f}")

## 5. ROC Curve & Save Models + Results

In [ ]:
# ROC curve — both versions on the same plot
fig, ax = plt.subplots(figsize=(7, 6))
for name, y_prob, color in [
    ('Scratch', y_prob_scratch, 'steelblue'),
    ('sklearn', y_prob_sk,      'darkorange')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.4f})', color=color, linewidth=2)
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Logistic Regression', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
roc_path = os.path.join(PLOTS_DIR, 'roc_logistic_regression.png')
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
plt.show()

# Save models
with open(os.path.join(MODELS_DIR, 'lr_scratch.pkl'), 'wb') as f:
    pickle.dump(lr_scratch, f)
with open(os.path.join(MODELS_DIR, 'lr_sklearn.pkl'), 'wb') as f:
    pickle.dump(lr_sk, f)
print("Models saved.")

# Save results (both versions)
save_results(metrics_scratch)
save_results(metrics_sk)